# Class 1: Matrix Foundations & Transformations
Run each cell with **Shift + Enter**. Change the numbers and run again!

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, size=5):
    """Show one or more images side by side (handles BGR colour and grayscale)."""
    titles = titles or [""] * len(images)
    plt.figure(figsize=(size * len(images), size))
    for k, (im, t) in enumerate(zip(images, titles), 1):
        plt.subplot(1, len(images), k)
        if im.ndim == 2:
            plt.imshow(im, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.title(t); plt.axis("off")
    plt.show()

## 1. How computers see (00–20 min)
A grayscale image is a 2D grid of numbers from 0 (black) to 255 (white).

In [ ]:
tiny = np.array([[  0,  50, 100],
                 [150, 200, 255]], dtype=np.uint8)
print(tiny.shape, tiny.dtype)
print("row 1, col 2 =", tiny[1, 2])
show(tiny, titles=["2 x 3 grayscale image"], size=4)

A colour image is 3D: `(height, width, 3)`. **OpenCV uses BGR order**, not RGB.

In [ ]:
canvas = np.zeros((200, 300, 3), dtype=np.uint8)   # black
canvas[:, :] = (255, 0, 0)                          # BLUE, because BGR
canvas[50:150, 100:200] = (0, 255, 255)             # yellow box
cv2.circle(canvas, (150, 100), 30, (0, 0, 255), -1) # red dot: centre is (x, y)
print(canvas.shape, "pixel at [100, 150] =", canvas[100, 150])
show(canvas, titles=["made from numbers"])

## 2. Basic transformations (20–45 min)

### Get a photo
Run **one** of the two cells below: A downloads a sample photo, B lets you upload your own.

In [ ]:
# A) sample photo
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/fruits.jpg", "photo.jpg")
path = "photo.jpg"

In [ ]:
# B) upload your own (Colab only)
from google.colab import files
uploaded = files.upload()
path = list(uploaded)[0]

In [ ]:
img = cv2.imread(path)
assert img is not None, "Could not read the image. Check the file name!"
print("shape:", img.shape, " dtype:", img.dtype)
show(img, titles=["original"])

### Colour spaces

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("colour:", img.shape, "  gray:", gray.shape)
wrong = img[:, :, ::-1]   # what you'd see if you forgot BGR -> RGB
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(img); plt.title("plt.imshow(BGR) - wrong colours!"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(gray, cmap="gray"); plt.title("grayscale"); plt.axis("off")
plt.show()

### Resize  (careful: `cv2.resize` takes **(width, height)**)

In [ ]:
small = cv2.resize(img, (200, 200))                      # squashed
h, w = img.shape[:2]
new_w = 200
fit = cv2.resize(img, (new_w, int(h * new_w / w)), interpolation=cv2.INTER_AREA)
print("original", img.shape, "| forced", small.shape, "| keep ratio", fit.shape)
show(small, fit, titles=["forced 200x200", "aspect ratio kept"], size=4)

### Rotate & flip: the rotation is a **2×3 matrix**

In [ ]:
r90 = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
flipped = cv2.flip(img, 1)
M = cv2.getRotationMatrix2D((w // 2, h // 2), 30, 1.0)
print("rotation matrix M =\n", M.round(3))
rot30 = cv2.warpAffine(img, M, (w, h))
show(r90, flipped, rot30, titles=["90° clockwise", "mirror", "30°"], size=4)

### Crop = slicing `img[y1:y2, x1:x2]`

In [ ]:
roi = img[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
print("roi shape:", roi.shape)
show(img, roi, titles=["original", "cropped centre"])

## 3. Enhancement & edges (45–75 min)
### Gaussian blur: kernel size must be odd

In [ ]:
noise = np.random.normal(0, 25, gray.shape)
noisy = np.clip(gray + noise, 0, 255).astype(np.uint8)
blur5 = cv2.GaussianBlur(noisy, (5, 5), 0)
blur15 = cv2.GaussianBlur(noisy, (15, 15), 0)
show(noisy, blur5, blur15, titles=["noisy", "5x5", "15x15"], size=4)

### Thresholding

In [ ]:
_, bw = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
T, otsu = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("Otsu chose T =", T)
show(bw, otsu, titles=["T = 127", f"Otsu T = {T:.0f}"])

### Canny edges: blur first, then try different low/high values

In [ ]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)
show(cv2.Canny(gray, 50, 150), cv2.Canny(blur, 50, 150), cv2.Canny(blur, 100, 250),
     titles=["no blur", "blur, 50/150", "blur, 100/250"], size=4)

## 4. Lab → open `lab01_edge_sketch.ipynb`